# Predict wins & losses for 25' season

In [ ]:
from meml_griz import fetch
import pandas as pd
import numpy as np
from datetime import datetime

### Fetch all NBA team data

In [ ]:
for team in (fetch.NbaTeams.MEM,):
    seasons = fetch.SeasonData.from_date_range(
        team=team,
        start_date=datetime(2021, 10, 1),
        end_date=datetime(2026, 5, 1),
    )

Clean data

In [ ]:
columns = seasons.team_stats_df.columns.tolist()

drops = {
    "gameId",
    "teamId",
    "teamCity",
    "teamName",
    "teamSlug",
    "gameDate",
    "teamTricode",
}
bools = {"home_game"}
cats = set()
targets = {"win"}
nums = set(columns) - drops - bools - targets

Make training and testing data

In [ ]:
train = seasons.team_stats_df.query("seasonYear < 2025")
test = seasons.team_stats_df.query("seasonYear == 2025")

### Make sklearn pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

number_cols = list(nums)
bool_cols = list(bools)

number_tx = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", number_tx, number_cols),
        ("bool_cols", "passthrough", bool_cols),
    ],
    remainder="drop",
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=99)),
    ]
)

Fit the data on win or lose

In [ ]:
X = train.drop(columns=list(targets))
y = train[list(targets)]

model = pipeline.fit(X, y)

## Predict win/loss

In [ ]:
predict = model.predict(test.drop(columns=list(targets)))
predict.sum()